In [ ]:
# ДЗ №1 Python
# Исаков Александр Андреевич, РИМ-150950


## Задание 1
Реализуйте метакласс ThreadSafeSingleton, который обеспечивает создание только одного экземпляра класса, даже в многопоточной среде.
Используйте `from threading import Lock`

# Решение

SyntaxError: invalid syntax (379367644.py, line 2)

In [7]:
from threading import Lock 
# импорт класса Lock из модуля threading

class ThreadSafeSingleton(type):
    _instances = {}
    _lock = Lock()

    def __call__(cls, *args, **kwargs):
        with cls._lock:
            if cls not in cls._instances:
                cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

In [8]:
class DatabasePool(int, metaclass = ThreadSafeSingleton):
    def __init__(self):
        self._connection = 'connected'
    def get_connection(self):
        return self._connection

In [9]:
# Проверка

In [11]:
# Создайте 3 экземпляра DatabasePool
pool1 = DatabasePool()
pool2 = DatabasePool()
pool3 = DatabasePool()

# Убедитесь, что это один и тот же объект
assert pool1 is pool2 is pool3

# Проверьте, что соединения разделяются между экземплярами
conn1 = pool1.get_connection()
pool2._connection = 'disconnected'
conn2 = pool2.get_connection()

In [ ]:
## Задание 2

Создайте метакласс, который считает, сколько раз создавался каждый класс.

Требования:
1. Метакласс должен иметь атрибут _counters
2. При создании экземпляра класса счетчик должен увеличиваться
3. Добавьте метод get_count(), который возвращает количество созданных экземпляров

## Решение

In [12]:
class CountInstancesMeta(type):
    _counters = {}

    def __call__(cls, *args, **kwargs): 
        # Создаётся экземпляр
        instance = super().__call__(*args, **kwargs) 
        # Увеличивается счётчик для класса
        cls._counters[cls] = cls._counters.get(cls, 0) + 1
        return instance

    def get_count(cls): 
        # Возвращается количество созданных экземпляров класса cls
        return cls._counters.get(cls, 0)

In [13]:
class User(metaclass = CountInstancesMeta):
    def __init__(self, name):
        self.name = name

class Product(metaclass = CountInstancesMeta):
    def __init__(self, name):
        self.name = name

In [14]:
# Проверка
user1 = User("Alice")
user2 = User("Bob")
product1 = Product("Laptop")

print(User.get_count())    # Должно быть 2
print(Product.get_count()) # Должно быть 1

2
1


In [ ]:
## Задание 3

Создайте метакласс, который автоматически добавляет метод describe() в каждый класс.

Требования:
1. Метод describe() должен возвращать строку с именем класса
2. Используйте метакласс для создания классов Car и Book

## Решение

In [15]:
class DescribeMeta(type):
    def __new__(cls, name, bases, attrs):
        def describe(self):
            return f"Это объект класса {self.__class__.__name__}"
        attrs['describe'] = describe
        return super().__new__(cls, name, bases, attrs)

In [16]:
class Car(metaclass = DescribeMeta):
    def __init__(self, brand):
        self.brand = brand

class Book(metaclass = DescribeMeta):
    def __init__(self, title):
        self.title = title

In [17]:
car = Car("Mazda")
book = Book("Как все понять-то?")

print(car.describe())  # Должно быть "Это объект класса Car"
print(book.describe()) # Должно быть "Это объект класса Book"

Это объект класса Car
Это объект класса Book


In [ ]:
## Задание 4

Создайте метакласс, который проверяет, что у класса есть метод save(). Можно использовать `__new__`

Требования:
1. Если у класса нет метода save(), метакласс должен выдать ошибку
2. Создайте класс User с методом save()
3. Попробуйте создать класс Message без метода save() (должна быть ошибка)

## Решение

In [18]:
class SaveMeta(type):
    def __new__(cls, name, bases, attrs):
        if 'save' not in attrs:
            raise TypeError(f"Класс {name} должен содержать метод save()")
        return super().__new__(cls, name, bases, attrs)

In [20]:
class User(metaclass=SaveMeta):
    # Класс с User с методом save()
    def __init__(self, name):
        self.name = name

    def save(self):
        print(f"Пользователь {self.name} сохранён")



In [21]:
# Проверка
user = User("Alice")
user.save()  # Должно работать

Пользователь Alice сохранён


In [22]:
class Message(metaclass = SaveMeta):
    def __init__(self, text):
        self.text = text

TypeError: Класс Message должен содержать метод save()